In [0]:

# MAGIC # 03 · Silver Layer — Clean, Dedupe, SCD Type 2
#1. Parse `raw_json` into typed columns per resource (schema below).

#2. **Dedupe** same-day duplicate pulls: keep the row with the max
#     `last_updated` per `resource_id`.

#**SCD2 MERGE**: compare an MD5 hash of the tracked columns
# against the current row in Silver.



dbutils.widgets.text("resource_type", "Patient")
resource_type = dbutils.widgets.get("resource_type")


import sys, os
sys.path.append(os.path.abspath("../common"))
from config import bronze_table, silver_table
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

src_table = bronze_table(resource_type)
tgt_table = silver_table(resource_type)


#  Resource-specific parsing
#  Each FHIR resource has a different shape, so we extract a small,
# report-friendly set of fields per type. `raw_json` is preserved
# alongside the flattened columns for auditability / re-parsing.


def parse_patient(df):
    return (
        df.withColumn("parsed", F.from_json("raw_json", _patient_schema()))
          .select(
              F.col("parsed.id").alias("resource_id"),
              F.col("parsed.meta.versionId").alias("version_id"),
              F.col("parsed.meta.lastUpdated").alias("last_updated"),
              F.col("parsed.gender").alias("gender"),
              F.col("parsed.birthDate").alias("birth_date"),
              F.element_at("parsed.name.family", 1).alias("family_name"),
              F.element_at(F.element_at("parsed.name.given", 1), 1).alias("given_name"),
              F.element_at("parsed.address.city", 1).alias("city"),
              F.element_at("parsed.address.state", 1).alias("state"),
              F.element_at("parsed.address.country", 1).alias("country"),
              F.col("raw_json"),
          )
    )

def parse_encounter(df):
    return (
        df.withColumn("parsed", F.from_json("raw_json", _encounter_schema()))
          .select(
              F.col("parsed.id").alias("resource_id"),
              F.col("parsed.meta.versionId").alias("version_id"),
              F.col("parsed.meta.lastUpdated").alias("last_updated"),
              F.col("parsed.status").alias("status"),
              F.col("parsed.class.code").alias("class_code"),
              F.regexp_extract("parsed.subject.reference", r"Patient/(.+)", 1).alias("patient_id"),
              F.col("parsed.period.start").alias("period_start"),
              F.col("parsed.period.end").alias("period_end"),
              F.col("raw_json"),
          )
    )

def parse_observation(df):
    return (
        df.withColumn("parsed", F.from_json("raw_json", _observation_schema()))
          .select(
              F.col("parsed.id").alias("resource_id"),
              F.col("parsed.meta.versionId").alias("version_id"),
              F.col("parsed.meta.lastUpdated").alias("last_updated"),
              F.col("parsed.status").alias("status"),
              F.element_at("parsed.code.coding.code", 1).alias("code"),
              F.element_at("parsed.code.coding.display", 1).alias("code_display"),
              F.regexp_extract("parsed.subject.reference", r"Patient/(.+)", 1).alias("patient_id"),
              F.regexp_extract("parsed.encounter.reference", r"Encounter/(.+)", 1).alias("encounter_id"),
              F.col("parsed.effectiveDateTime").alias("effective_datetime"),
              F.col("parsed.valueQuantity.value").alias("value_quantity"),
              F.col("parsed.valueQuantity.unit").alias("value_unit"),
              F.col("raw_json"),
          )
    )

def parse_condition(df):
    return (
        df.withColumn("parsed", F.from_json("raw_json", _condition_schema()))
          .select(
              F.col("parsed.id").alias("resource_id"),
              F.col("parsed.meta.versionId").alias("version_id"),
              F.col("parsed.meta.lastUpdated").alias("last_updated"),
              F.element_at("parsed.clinicalStatus.coding.code", 1).alias("clinical_status"),
              F.element_at("parsed.code.coding.code", 1).alias("code"),
              F.element_at("parsed.code.coding.display", 1).alias("code_display"),
              F.regexp_extract("parsed.subject.reference", r"Patient/(.+)", 1).alias("patient_id"),
              F.regexp_extract("parsed.encounter.reference", r"Encounter/(.+)", 1).alias("encounter_id"),
              F.col("parsed.onsetDateTime").alias("onset_datetime"),
              F.col("parsed.recordedDate").alias("recorded_date"),
              F.col("raw_json"),
          )
    )


 ### Minimal FHIR schemas (only fields we project — keeps parsing fast)


def _meta_field():
    return "STRUCT<versionId: STRING, lastUpdated: STRING>"

def _patient_schema():
    return f"""
      STRUCT<
        id: STRING, meta: {_meta_field()}, gender: STRING, birthDate: STRING,
        name: ARRAY<STRUCT<family: STRING, given: ARRAY<STRING>>>,
        address: ARRAY<STRUCT<city: STRING, state: STRING, country: STRING>>
      >"""

def _encounter_schema():
    return f"""
      STRUCT<
        id: STRING, meta: {_meta_field()}, status: STRING,
        class: STRUCT<code: STRING>,
        subject: STRUCT<reference: STRING>,
        period: STRUCT<start: STRING, end: STRING>
      >"""

def _coding_array():
    return "STRUCT<coding: ARRAY<STRUCT<code: STRING, display: STRING>>>"

def _observation_schema():
    return f"""
      STRUCT<
        id: STRING, meta: {_meta_field()}, status: STRING,
        code: {_coding_array()},
        subject: STRUCT<reference: STRING>,
        encounter: STRUCT<reference: STRING>,
        effectiveDateTime: STRING,
        valueQuantity: STRUCT<value: DOUBLE, unit: STRING>
      >"""

def _condition_schema():
    return f"""
      STRUCT<
        id: STRING, meta: {_meta_field()},
        clinicalStatus: {_coding_array()},
        code: {_coding_array()},
        subject: STRUCT<reference: STRING>,
        encounter: STRUCT<reference: STRING>,
        onsetDateTime: STRING, recordedDate: STRING
      >"""

PARSERS = {
    "Patient": parse_patient, "Encounter": parse_encounter,
    "Observation": parse_observation, "Condition": parse_condition,
}
TRACKED_COLS = {
    "Patient": ["gender", "birth_date", "family_name", "given_name", "city", "state", "country"],
    "Encounter": ["status", "class_code", "patient_id", "period_start", "period_end"],
    "Observation": ["status", "code", "code_display", "patient_id", "encounter_id",
                     "effective_datetime", "value_quantity", "value_unit"],
    "Condition": ["clinical_status", "code", "code_display", "patient_id",
                  "encounter_id", "onset_datetime", "recorded_date"],
}


 ### Load today's bronze increment, parse, dedupe


bronze_today = spark.table(src_table).filter(
    F.col("ingestion_date") == F.current_date()
)

if bronze_today.limit(1).count() == 0:
    print(f"No new bronze rows for {resource_type} today — skipping silver merge.")
    dbutils.notebook.exit("NO_DATA")

parsed = PARSERS[resource_type](bronze_today)
tracked = TRACKED_COLS[resource_type]

# Dedupe: within this batch, keep the latest version per resource_id
w = F.row_number().over(
    Window.partitionBy("resource_id").orderBy(F.col("last_updated").desc())
)
deduped = parsed.withColumn("_rn", w).filter("_rn = 1").drop("_rn")

hashed = deduped.withColumn(
    "row_hash", F.md5(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in tracked]))
)


 ### SCD Type 2 MERGE into Silver
# Two-step Delta pattern (standard SCD2-via-MERGE):
#  1. MERGE that only **expires** changed current rows.
#  2. Append the new/changed rows as fresh current versions.
#  (A single MERGE can't both update-and-insert two output rows for
#  the same key, hence the two-step.)

scd_cols = ["resource_id", "version_id", "last_updated", "row_hash",
            "effective_start_date", "effective_end_date", "is_current"] + tracked + ["raw_json"]

if not spark.catalog.tableExists(tgt_table):
    (hashed
        .withColumn("effective_start_date", F.current_timestamp())
        .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .select(*scd_cols)
        .write.format("delta").mode("overwrite")
        .saveAsTable(tgt_table))
    print(f"[{resource_type}] created silver table with {hashed.count()} initial rows")
else:
    delta_tgt = DeltaTable.forName(spark, tgt_table)
    staged = hashed.withColumn("effective_start_date", F.current_timestamp()) \
                    .withColumn("effective_end_date", F.lit(None).cast("timestamp")) \
                    .withColumn("is_current", F.lit(True)) \
                    .select(*scd_cols)
    staged.createOrReplaceTempView("staged_updates")

    # Step 1: expire current rows whose hash changed
    delta_tgt.alias("t").merge(
        staged.alias("s"),
        "t.resource_id = s.resource_id AND t.is_current = true"
    ).whenMatchedUpdate(
        condition="t.row_hash <> s.row_hash",
        set={
            "is_current": "false",
            "effective_end_date": "current_timestamp()",
        },
    ).execute()

    # Step 2: insert new-current rows for new ids OR ids whose hash changed
    current_ids_unchanged = (
        spark.table(tgt_table)
        .filter("is_current = true")
        .select("resource_id", "row_hash")
    )
    to_insert = staged.join(
        current_ids_unchanged, on="resource_id", how="left_anti"
    )
    to_insert.write.format("delta").mode("append").saveAsTable(tgt_table)
    print(f"[{resource_type}] SCD2 merge: {to_insert.count()} new/changed current row(s)")

dbutils.jobs.taskValues.set(key="silver_rows_processed", value=deduped.count())
